# 03 — B0 / B1 baselines (paper §4.1 risky test)

Phase 1 의 SimBench argmax 71.2% 가 *MoE-KG-cycle architecture 의 contribution* 인지 검증.

**B0** = Generic Encoder + capacity-matched MLP (~50M) + FactDecoder — no MoE.
**B1** = Generic Encoder + Standard MoE (Switch top-1, k=20) + FactDecoder — no shared experts, no sparsegen.

같은 cycle pretrain setup (batch=32, lr=1e-4, λ_lb=0.1, epoch=30) — Phase 1 의 ph1_v3_minimal 와 fair.

**Decision criterion**:
- B0 SimBench argmax > 0.65 → BGE-large + linear head 만으로도 충분. paradigm contribution 약화.
- B0/B1 SimBench argmax < 0.60 → +10pp 이상의 Phase 1 contribution. paradigm 확정.

In [ ]:
# Pre-setup — Colab kernel 마다 첫 cell.
import os, sys
BASE = '/content/drive/MyDrive/sideproject'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('cwd =', os.getcwd())

## A. B0 cycle pretrain (~55분)

GenericBaseline = no MoE. capacity-matched MLP (~50M params) + FactDecoder. Auto-resume 작동.

In [ ]:
from phase1.baselines.train_baselines import train_baseline

train_baseline(
    baseline='B0',
    run_id='b0_v0',
    epochs=30,
    batch_size=32,
    lr=1e-4,
    warmup_steps=300,
    grad_clip=1.0,
    mlp_width=4500,
    mlp_n_hidden=3,
)

## B. B1 cycle pretrain (~55분)

StandardMoEBaseline = Switch top-1 routing, k_routed=20 (Phase 1 의 16 routed + 4 shared 의 capacity match). λ_lb=0.1 동일.

In [ ]:
from phase1.baselines.train_baselines import train_baseline

train_baseline(
    baseline='B1',
    run_id='b1_v0',
    epochs=30,
    batch_size=32,
    lr=1e-4,
    warmup_steps=300,
    grad_clip=1.0,
    k_routed=20,
    d_hidden=2048,
    lambda_lb=0.1,
)

## C. SimBench Path A eval (B0 + B1)

같은 classifier head spec (hidden=512, dropout=0.1, lr=1e-3) + epoch=100 — Phase 1 와 fair. feature cache 가 모델별로 분리됨 (`features_{run_id}.npz`).

In [ ]:
from phase1.eval_simbench_classifier import train_simbench_classifier

for run_id in ['b0_v0', 'b1_v0']:
    print(f'\n{"="*70}\nSimBench Path A — {run_id}\n{"="*70}')
    train_simbench_classifier(
        run_id=run_id,
        epochs=100,
        batch_size=256,
        lr=1e-3,
        hidden=512,
        dropout=0.1,
    )

## D. 비교 — Phase 1 vs B0 vs B1

**Risky test verdict**: Phase 1 의 test_acc 가 B0/B1 보다 의미 있게 (예: +5pp) 높으면 paradigm contribution 확정. 비슷하거나 낮으면 narrative 조정 필요.

In [ ]:
import json
from pathlib import Path

print(f'{"run_id":<20} {"type":<7} {"best_ep":>8} {"val_KL":>8} {"test_KL":>8} {"test_acc":>9}')
print('-' * 65)
for run_id in ['ph1_v3_minimal', 'b0_v0', 'b1_v0']:
    p = Path(f'out/phase1/{run_id}/simbench_eval.json')
    if not p.exists():
        print(f'{run_id:<20}  MISSING'); continue
    r = json.loads(p.read_text())
    print(f'{run_id:<20} {r.get("model_type", "phase1"):<7} '
          f'{r["best_epoch"]:>8} '
          f'{r["best_val_kl"]:>8.4f} '
          f'{r["test"]["kl"]:>8.4f} '
          f'{r["test"]["argmax_acc"]:>9.4f}')